# 02 — Feature Analysis

Evaluate the engineered features and understand what drives model performance.

Sections:
1. Feature importance (LightGBM)
2. Target encoding distributions — do email domains separate fraud?
3. Velocity features — transaction count vs fraud rate
4. Cyclical time features — visualising sin/cos encoding
5. SMOTE before/after — visualising class rebalancing

**LinkedIn post angle:** *'The feature that moved my fraud model's PR-AUC by 8 points.'*

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

DATA_DIR = Path('../data')
SRC_DIR = Path('..')

## 1. Load engineered features

In [ ]:
import sys
sys.path.append(str(SRC_DIR))

from src.features.build_features import build_features

df = pd.read_parquet(DATA_DIR / 'processed/merged.parquet')
# Use a sample for notebook speed
df_sample = df.sample(50_000, random_state=42)

X, y, encoders = build_features(df_sample, fit=True)
print(f'Feature matrix: {X.shape}')
print(f'Feature list (first 20): {list(X.columns[:20])}')

## 2. LightGBM feature importance

In [ ]:
from lightgbm import LGBMClassifier

X_filled = X.fillna(-999)
model = LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
model.fit(X_filled, y)

importance = pd.DataFrame({
    'feature': X.columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).head(30)

fig = px.bar(
    importance,
    x='importance',
    y='feature',
    orientation='h',
    title='Top 30 Feature Importances (LightGBM, gain)',
)
fig.update_layout(height=700, yaxis={'categoryorder': 'total ascending'})
fig.show()

## 3. Email domain fraud rates — target encoding

In [ ]:
domain_fraud = df_sample.groupby('P_emaildomain')['isFraud'].agg(['mean', 'count']).reset_index()
domain_fraud.columns = ['domain', 'fraud_rate', 'count']
domain_fraud = domain_fraud[domain_fraud['count'] > 100].sort_values('fraud_rate', ascending=False)

fig = px.bar(
    domain_fraud.head(20),
    x='domain',
    y='fraud_rate',
    title='Fraud Rate by Purchaser Email Domain (min 100 transactions)',
    labels={'fraud_rate': 'Fraud Rate', 'domain': 'Email Domain'}
)
fig.show()
# This shows exactly what gets encoded into the target-encoded feature.

## 4. Cyclical time features visualisation

In [ ]:
hours = np.arange(0, 24)
hour_sin = np.sin(2 * np.pi * hours / 24)
hour_cos = np.cos(2 * np.pi * hours / 24)

fig = go.Figure()
fig.add_trace(go.Scatter(x=hours, y=hour_sin, name='sin(hour)', mode='lines+markers'))
fig.add_trace(go.Scatter(x=hours, y=hour_cos, name='cos(hour)', mode='lines+markers'))
fig.update_layout(
    title='Cyclical Encoding of Hour of Day',
    xaxis_title='Hour',
    yaxis_title='Encoded value',
    annotations=[{
        'x': 11.5, 'y': -1.1,
        'text': 'Hour 23 and hour 0 are close in this space (both near sin≈0, cos≈1)',
        'showarrow': False
    }]
)
fig.show()